# 123の直接設計を改善し、同じ設計を提案法へ適用する

上から順に実行します。従来15モード・75 µm間隔と、広帯域43モード・37.5 µm間隔、干渉を考慮した係数設計を比較します。まず2D初期場・FFTを作らず解析シートで候補を比較し、次に選んだ候補だけ有限開口・CGH再生・提案法で検証します。

AUTOは解析的な強度相対L2誤差が最小の候補を選びます。有限開口の結果でも改善するかは最後の比較で判断します。従来の結果フォルダは残します。詳しくは `LSH数字123_画質改善.md`。

広帯域案では中心モード半径30 µm、開口半径約5.8 mm、仮想4f出力面の座標間隔7 µmを用います。実機SLMの倍率を校正した設定ではありません。波数とキャリアの整合性は設定表で確認します。

**広帯域案は、先に lsh_digits123_rank_audit.ipynb で圧縮基底を検証してください。ランク100固定での再伝搬を防ぎ、検証した親基底と列をそのまま使います。**


In [ ]:
import importlib
import json
import hashlib
import csv
from dataclasses import replace
from pathlib import Path
import numpy as np
import lsh_digit1_theory as theory
import lsh_digit1_proposed as proposed
import lsh_digits123_design as design_study
theory = importlib.reload(theory)
proposed = importlib.reload(proposed)
design_study = importlib.reload(design_study)

# 主な変更箇所。長さは50 cmを維持します。
RANK = 100  # USE_VERIFIED_RANK=Trueなら検証済みの値へ変更する
MAX_INITIAL_COMPLEX_ERROR = 0.01
USE_VERIFIED_RANK = True
LIBRARY_PARENT_RANK = None
SEED = 42
RUN_VERSION = 'v1'
DESIGN_CASE = 'auto'  # 'wide_dense_coupled'などを指定すると候補を固定
CHECK_PADDING = False  # True: 有限開口伝搬のpadding 2/3比較も行う
base = replace(theory.Digit1Config(), target_digits='123', digit_axes='zx',
    coefficient_samples=4096, coefficient_fit_x_samples=401,
    coefficient_ridge_relative=1e-3, z_samples=251, check_padding=CHECK_PADDING)
candidates = design_study.design_candidates(base)
screen_dir = Path(f'lsh_digits123_design_outputs_{RUN_VERSION}')
for name, candidate in candidates.items():
    report = design_study.design_report(candidate)
    print(name, json.dumps(report, ensure_ascii=False))
print('次のセルで解析候補比較、その次のセルで初期場・圧縮辞書・CGH・伝搬を計算します。')


In [ ]:
# 解析式で5条件を比較。CGH/FFT/Tuckerを計算する前に直接設計を確認する。
from IPython.display import Image, display
screen = design_study.analytic_screen(screen_dir, candidates)
display(Image(filename=str(screen_dir/'analytic_design_comparison.png'), width=1100))
for row in screen['cases']:
    print(row['case'], '強度相対L2:', row['intensity_relative_l2'],
          'SSIM:', row['ssim'], 'PSNR[dB]:', row['psnr_db'],
          '目標が明るい領域の誤差:', row['foreground_intensity_relative_l2'])
chosen = screen['best_analytic_case'] if DESIGN_CASE == 'auto' else DESIGN_CASE
if chosen not in candidates:
    raise ValueError(f'不明な条件: {chosen}')
config = candidates[chosen]
recommendation = None
if USE_VERIFIED_RANK:
    audit_file = Path('lsh_digits123_rank_audit_outputs_v1/summary.json')
    if not audit_file.is_file():
        raise ValueError('まずlsh_digits123_rank_audit.ipynbを実行し、圧縮場のランクを検証してください。')
    checked = json.loads(audit_file.read_text(encoding='utf-8'))
    if (checked.get('study') != 'digits123_shared_basis_initial_rank_audit'
            or theory.config_payload(checked.get('config', {})) != theory.config_payload(config)):
        raise ValueError('ランク検証と今回の光学条件が異なります。同じ条件で検証してください。')
    recommendation = checked.get('recommendation')
    if recommendation is None:
        raise ValueError('合格ランクがありません。CGH・伝搬へ進む前に基底を見直してください。')
    if recommendation.get('mode_spacing_factor', 1) != 1:
        raise ValueError('波数間隔が解析比較の条件と異なります。')
    SEED = recommendation.get('seed', SEED)
    RANK = recommendation['rank']
    LIBRARY_PARENT_RANK = recommendation['library_parent_rank']
    print('検証済みランク:', RANK, '親基底ランク:', LIBRARY_PARENT_RANK)
print('選んだ条件:', chosen)
print('注意：解析式での選択です。有限開口・CGHの結果は次のセルで検証します。')
# 辞書名に物理条件とThread配置の署名を入れ、旧辞書の誤使用を防ぐ。
spec = proposed.library_spec(config, LIBRARY_PARENT_RANK or RANK, SEED, basis_strategy='shift_aware')
signature = hashlib.sha256(json.dumps(spec, sort_keys=True).encode('utf-8')).hexdigest()[:16]
library_path = Path(recommendation['library_path']) if recommendation else Path(f'lsh_digits123_design_library_rank{RANK}_{signature}.npz')
theory_dir = Path(f'lsh_digits123_improved_theory_{chosen}_outputs_{RUN_VERSION}')
proposed_dir = Path(f'lsh_digits123_improved_proposed_{chosen}_outputs_rank{RANK}_parent{LIBRARY_PARENT_RANK or RANK}_{RUN_VERSION}')
print('圧縮辞書:', library_path.resolve())
print('理論結果:', theory_dir.resolve())
print('比較結果:', proposed_dir.resolve())


In [ ]:
# 選んだ条件で理論計算と提案法を検証。同じ係数・kr・Thread・CGH条件を使う。
expected = theory.config_payload(config)
summaries = {}
for kind, folder in (('theory', theory_dir), ('proposed', proposed_dir)):
    summary_file = folder/'summary.json'
    if summary_file.is_file() and (folder/'manifest.json').is_file():
        data = json.loads(summary_file.read_text(encoding='utf-8'))
        study = 'theory_only_digit1_single_xz_sheet' if kind == 'theory' else 'digit1_direct_vs_projected_shared_core'
        version = theory.THEORY_SCHEMA_VERSION if kind == 'theory' else proposed.SCHEMA_VERSION
        valid = (data.get('study') == study and data.get('schema_version') == version
                 and theory.config_payload(data.get('config', {})) == expected)
        if kind == 'proposed':
            valid = valid and (data.get('rank') == RANK and data.get('seed') == SEED
                and data.get('basis_strategy') == 'shift_aware'
                and data.get('mode_spacing_factor', 1) == 1
                and data.get('library_path') == str(library_path.resolve())
                and data.get('library_parent_rank', data.get('rank')) == (LIBRARY_PARENT_RANK or RANK))
        if not valid:
            raise ValueError(f'{folder}: 別条件です。RUN_VERSIONを変更してください。')
        if kind == 'proposed' and data['metrics']['initial_complex_proposed_vs_direct']['complex_relative_l2'] > MAX_INITIAL_COMPLEX_ERROR:
            raise ValueError('保存された圧縮場が基準を満たしていません。ランク検証を先に行ってください。')
        print('同じ条件の保存結果を読み込み:', folder)
    elif folder.exists() and any(folder.iterdir()):
        raise ValueError(f'{folder}: 途中結果です。RUN_VERSIONを変更してください。')
    elif kind == 'theory':
        data = theory.run_digit1(folder, config)
    else:
        data = proposed.run_comparison(folder, config, rank=RANK, seed=SEED,
            library_path=library_path, basis_strategy='shift_aware', run_shift_diagnostics=True,
            max_initial_complex_error=MAX_INITIAL_COMPLEX_ERROR, library_parent_rank=LIBRARY_PARENT_RANK)
    summaries[kind] = data
    print(kind, '保存先:', folder.resolve())


In [ ]:
# 目標への再現性と、提案法/直接法の一致を別々に確認する。
display(Image(filename=str(theory_dir/'digit1_sheet_comparison.png'), width=1300))
for filename in ('target_direct_proposed.png', 'cgh_sheet_comparison.png',
                 'cgh_mask_comparison.png', 'initial_field_comparison.png'):
    display(Image(filename=str(proposed_dir/filename), width=1200))
result = summaries['proposed']
print('選択条件:', chosen, 'ランク:', result['rank'])
print('使用Thread数:', result['active_thread_count'], 'Bessel項数:', result['nonzero_bessel_term_count'])
print('直接法/目標:', result['metrics']['direct_cgh_sheet_vs_target'])
print('提案法/直接法:', result['metrics']['cgh_sheet_proposed_vs_direct'])
print('提案法/目標:', result['metrics']['proposed_cgh_sheet_vs_target'])
print('計算時間:', result['timings_seconds'])
print('共通の係数設計時間は場生成とは別に記録します。')
print('シフト誤差の内訳:', result['shift_diagnostics'])
print('提案CGH:', (proposed_dir/'proposed_cgh_8bit.png').resolve())


In [ ]:
# 以前の123と新しい結果を、保存強度の共通座標への補間で比較。
# 新しい伝搬計算ではありません。最適な明るさ合わせも行いません。
baseline_dir = Path('lsh_digits123_proposed_zx_outputs_L0p5_rank100_v1')
records = []
for label, folder in (('previous_123', baseline_dir), (chosen, proposed_dir)):
    if not (folder/'summary.json').is_file():
        print('保存結果なし:', folder)
        continue
    saved_config = theory.Digit1Config(**json.loads((folder/'summary.json').read_text(encoding='utf-8'))['config'])
    target_keys = ('target_digits', 'digit_axes', 'digit_gap_fraction', 'digit_width_m',
                   'digit_z_start_m', 'digit_z_end_m', 'design_length_m', 'evaluation_x_half_width_m')
    if any(getattr(saved_config, key) != getattr(config, key) for key in target_keys):
        print('目標・物理範囲が異なるため比較しません:', folder)
        continue
    for row in design_study.saved_replay_scores(folder):
        row = {'case': label, **row}
        records.append(row)
        print(label, row['method'], '共通座標での強度相対L2:', row['intensity_relative_l2'],
              'SSIM:', row['ssim'], 'PSNR[dB]:', row['psnr_db'],
              '1次光のパワー割合:', row['first_order_power_fraction_of_incident'])
if records:
    score_file = screen_dir/f'finite_replay_scores_{chosen}_rank{RANK}.csv'
    with score_file.open('w', encoding='utf-8-sig', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)
    print('保存:', score_file.resolve())
print('CGH表示までの時間を比べる場合、別NotebookのSLM計測にも同じconfig・辞書を指定してください。')
